# CTS Model Demo — M3 Verification

Demonstrates each Triton model end-to-end:
1. **person-detector** — YOLO11m: JPEG → DetectionBox list
2. **reid-solider** — SOLIDER-REID: crop → 768-dim embedding
3. **pose-rtmpose** — RTMPose-m: crop → 17 COCO keypoints

**Prerequisites**: Triton must be running (`docker compose up triton`) with
model files present in `triton-models/`.  See `triton-models/README.md`.

```bash
cd continuous-tracking
uv sync --extra triton
jupyter lab tracking-orchestrator/notebooks/model_demo.ipynb
```

In [ ]:
import sys
from pathlib import Path

import numpy as np

# Add orchestrator package root to path when running from notebooks/
repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

TRITON_URL = "localhost:8001"  # change if running Triton on a remote host

In [ ]:
# ---------------------------------------------------------------------------
# Helper: load a test image (create a synthetic one if no JPEG is available)
# ---------------------------------------------------------------------------
def make_test_image(h: int = 480, w: int = 640) -> np.ndarray:
    """Return a synthetic RGB image shaped (H, W, 3) uint8."""
    rng = np.random.default_rng(42)
    img = rng.integers(0, 255, (h, w, 3), dtype=np.uint8)
    # Draw a rough person silhouette rectangle
    img[100:400, 250:380] = [200, 180, 160]
    return img


test_image = make_test_image()
print(f"Test image: {test_image.shape} dtype={test_image.dtype}")

---
## 1. person-detector — YOLO11m

Input: full-frame RGB image  
Output: list of `DetectionBox` (x1, y1, x2, y2 normalised [0,1], confidence)

In [ ]:
from app.inference import PersonDetector, TritonGrpcClient


async def demo_detector(image: np.ndarray) -> None:
    async with TritonGrpcClient(TRITON_URL) as client:
        ready = await client.is_model_ready("person-detector")
        print(f"person-detector ready: {ready}")
        if not ready:
            print("  -> Run triton-models/scripts/export_yolo.py first.")
            return

        detector = PersonDetector(client)
        boxes = await detector.detect(image)
        print(f"Detected {len(boxes)} person(s):")
        for i, b in enumerate(boxes):
            coord = f"({b.x1:.3f},{b.y1:.3f})-({b.x2:.3f},{b.y2:.3f})"
            print(f"  [{i}] {coord}  conf={b.confidence:.3f}")


await demo_detector(test_image)

---
## 2. reid-solider — SOLIDER-REID

Input: person crop RGB (H, W, 3)  
Output: 768-dim L2-normalised embedding (float32)

In [ ]:
from app.inference import ReidEmbedder

# Synthetic person crop
crop_a = make_test_image(256, 128)[:, :, :]  # same person — should give similar embeddings
crop_b = make_test_image(256, 128)[:, :, :]  # slightly different


async def demo_reid(crop1: np.ndarray, crop2: np.ndarray) -> None:
    async with TritonGrpcClient(TRITON_URL) as client:
        ready = await client.is_model_ready("reid-solider")
        print(f"reid-solider ready: {ready}")
        if not ready:
            print("  → Run triton-models/scripts/export_reid.py first.")
            return

        embedder = ReidEmbedder(client)
        emb1, emb2 = await embedder.embed_batch([crop1, crop2])

        print(f"Embedding shape: {emb1.shape}  dtype={emb1.dtype}")
        print(f"Embedding norm (should be ~1.0): {np.linalg.norm(emb1):.4f}")
        cosine_sim = float(np.dot(emb1, emb2))
        print(f"Cosine similarity between two crops: {cosine_sim:.4f}")
        print("  (1.0 = identical, 0.0 = unrelated, negative = opposite appearance)")


await demo_reid(crop_a, crop_b)

---
## 3. pose-rtmpose — RTMPose-m

Input: person crop RGB (H, W, 3)  
Output: PoseResult with 17 COCO keypoints (x, y normalised [0,1], score)

In [ ]:
from app.inference import PoseEstimator
from app.inference.schemas import COCO_KEYPOINTS

person_crop = make_test_image(300, 150)


async def demo_pose(crop: np.ndarray) -> None:
    async with TritonGrpcClient(TRITON_URL) as client:
        ready = await client.is_model_ready("pose-rtmpose")
        print(f"pose-rtmpose ready: {ready}")
        if not ready:
            print("  -> Run triton-models/scripts/export_pose.py first.")
            return

        estimator = PoseEstimator(client)
        result = await estimator.infer(crop)

        print(f"{len(result.keypoints)} keypoints detected:")
        for name, kp in zip(COCO_KEYPOINTS, result.keypoints, strict=True):
            print(f"  {name:<20} x={kp.x:.3f}  y={kp.y:.3f}  score={kp.score:.3f}")


await demo_pose(person_crop)

---
## 4. Model readiness summary

Quick check confirming all three models are loaded. Expected output when
model files are present: three `True` lines.

This cell corresponds to the DoD gate:
> `curl :8000/v2/models/ready` returns ready for all three models.

In [ ]:
async def check_all_ready() -> None:
    models = ["person-detector", "reid-solider", "pose-rtmpose"]
    async with TritonGrpcClient(TRITON_URL) as client:
        for m in models:
            ready = await client.is_model_ready(m)
            status = "READY ✓" if ready else "NOT READY — see triton-models/README.md"
            print(f"  {m:<20} {status}")


await check_all_ready()